[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/main/notebooks/Bi-encoder_experiments_evaluation.ipynb)

### Load the model you want to evaluate:

In [ ]:
from sentence_transformers import SentenceTransformer

# This is the Bi-Encoder model used in RQ3
model_name = "Stevenf232/SapBERT_MultipleNegativesRankingLoss_BC5CDR_Context"
model = SentenceTransformer(model_name)

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'
model.to(device)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")
data = dataset["test"]

### Load dataset and pre-process the data:

In [ ]:
from datasets import Dataset

# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window

context_window = 128

mention_names = [p['mention'] for p in data]
seen_mention_texts = [get_mt_window(p['mention'], p['mention_text'], context_window) for p in data]
entity_names = [p['entity'] for p in data]
seen_definitions = [p['definition'][:128] for p in data]


def prepare_sentence_pair_dataset(dataset_entry, idx):
    """
    Prepares a single entry from the raw dataset into the format
    expected by SentenceTransformerTrainer using pre-calculated variables.
    """
    m = mention_names[idx]
    mt_window = seen_mention_texts[idx]
    e = entity_names[idx]
    d_processed = seen_definitions[idx]

    dataset_entry["sentence1"] = f"{m} [SEP] {mt_window}"
    dataset_entry["sentence2"] = f"{e} [SEP] {d_processed}"

    return dataset_entry

In [ ]:
test_dataset = data.map(
    prepare_sentence_pair_dataset,
    with_indices=True,
    # we only need the following columns:
    # sentences1 and 2 for encoding
    # entity, definition, and ID - we need to create an additional dataset
    # because we need to remove duplicate entities from the dataset
    # so that we don't compare against the same entity multiple times
    remove_columns=['mention', 'mention_text', 'aliases']
)

print(f"Prepared test_dataset sample: {test_dataset[0]}")

In [ ]:
# mentions
query_embeddings = model.encode(test_dataset["sentence1"], convert_to_tensor=True)

In [ ]:
# entities
# we need to remove duplicates so we don't compare against the same entity multiple times

import pandas as pd
from datasets import Dataset

# 1. Convert to Pandas to handle duplicates easily
df = test_dataset.to_pandas()

# 2. Remove duplicates based on the specific column
df_cleaned = df.drop_duplicates(subset=["entity"])

# 3. Convert back to a Hugging Face Dataset
entities = Dataset.from_pandas(df_cleaned)

# 4. Now encode the unique sentences
passage_embeddings = model.encode(entities["sentence2"], convert_to_tensor=True)

# Model evaluation - Accuracy@k

In [ ]:
from sentence_transformers import util

k = 5 # accuracy@k
scores = util.semantic_search(query_embeddings, passage_embeddings, top_k=k) # cosine similarity is the default scoring function

In [ ]:
def evaluate_at_k(scores, data, k=5):
    correct_at_1_count = 0
    correct_in_top_k_count = 0
    mrr_sum = 0.0
    total_queries = len(data)

    print(f"Evaluating at k={k}")

    for i, hits in enumerate(scores):
        found_in_top_k = False
        rank = 0

        # Get the correct ID for the current query
        correct_id = data[i]["id"]

        for j, hit in enumerate(hits): # hit is {'corpus_id': <index in entities>, 'score': <similarity score}
            rank = j + 1
            top_idx = hit['corpus_id']
            matched_entry = entities[top_idx]

            top_match_id = matched_entry["id"]

            # Check if this hit is the correct one
            if top_match_id == correct_id:
                found_in_top_k = True
                if rank == 1: # For accuracy@1, this is the top match
                    correct_at_1_count += 1
                # Increment correct_in_top_k_count if found within top k
                correct_in_top_k_count += 1
                mrr_sum += 1.0 / rank
                break # Found the correct one, no need to check further hits for this query

        # Prepare logging information for each query
        mention_name = data[i]["mention"]
        correct_entity_name = data[i]["entity"]
        seen_mention_text = seen_mention_texts[i]
        mention_text = data[i]["mention_text"]

        print(f"--- Query {i+1} ---")
        print(f"Mention: {mention_name}")
        print(f"Correct Entity: {correct_entity_name}")
        print(f"seen_mention_text: {seen_mention_text}")
        print(f"mention_text: {mention_text}")

        if found_in_top_k:
            print(f"Correct entity found at rank: {rank}")
        else:
            print(f"Correct entity NOT found in top {k} results.")
            print(f"Top {k} (wrong) matches:")
            for j, hit in enumerate(hits):
                top_idx = hit['corpus_id']
                matched_entry = entities[top_idx]
                top_match_entity = matched_entry['entity']
                top_match_definition = matched_entry['definition'].strip()
                print(f"  Rank {j+1}: Entity: {top_match_entity}: Definition: {top_match_definition}")

        print("\n")

    accuracy_at_1 = correct_at_1_count / total_queries
    accuracy_at_k = correct_in_top_k_count / total_queries # Calculate accuracy@k
    mean_reciprocal_rank = mrr_sum / total_queries

    print(f"Total queries: {total_queries}")
    print(f"Accuracy@1: {accuracy_at_1:.4f}")
    print(f"Accuracy@{k}: {accuracy_at_k:.4f}") # Print accuracy@k
    print(f"Mean Reciprocal Rank (MRR): {mean_reciprocal_rank:.4f}")

    return accuracy_at_1, accuracy_at_k, mean_reciprocal_rank

### Save to file:

In [ ]:
import contextlib
from pathlib import Path
import os



dir = "results"
file = f"Bi_encoder_SapBert_at_{k}.txt"
path = os.path.join(dir, file)

# Creates the folder if it's missing; does nothing if it already exists
Path(dir).mkdir(parents=True, exist_ok=True)

with open(path, 'w') as f:
    with contextlib.redirect_stdout(f):
        evaluate_at_k(scores, data, k=k)
